In [ ]:
# In[1]:

import pandas as pd

# load metric data
df = pd.read_csv('metric.csv')

# parse timestamp as UTC datetime (per rules)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# ensure value is numeric
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# aggregate global statistics per (cmdb_id, kpi_name) using the entire series
grp = df.groupby(['cmdb_id', 'kpi_name'])['value']
agg = grp.agg(
    count='count',
    p95=lambda x: x.quantile(0.95),
    mean='mean',
    max='max'
).reset_index()

# sort by count descending and limit to top 50 rows
agg_sorted = agg.sort_values('count', ascending=False)
agg_top50 = agg_sorted.head(50)

# short list of unique cmdb_id values (limit to first 20)
unique_cmdbs_df = pd.DataFrame({'cmdb_id': pd.unique(df['cmdb_id'])[:20]})

# Display the two compact results
agg_top50, unique_cmdbs_df

```
Out[1]:
```
```python
# Build a plain-English summary string based on prior results in the IPython state
unique_count = unique_cmdbs_df.shape[0]
unique_list = ', '.join(unique_cmdbs_df['cmdb_id'].tolist())

sample = agg_top50.head(5)[['cmdb_id','kpi_name','count','p95','mean','max']].copy()
sample[['p95','mean','max']] = sample[['p95','mean','max']].applymap(lambda x: f"{x:.6g}")
sample_str = sample.to_string(index=False)

summary = (
    f"Summary of metric.csv global statistics (full series):\n"
    f"- Unique components found: {unique_count}\n"
    f"- Components (up to first 20): {unique_list}\n"
    f"- Computed per-(cmdb_id, kpi_name) aggregates: count, p95, mean, max using the entire time series.\n"
    f"- Returned up to top 50 rows sorted by sample count; showing top 5 aggregated rows below:\n\n"
    f"{sample_str}\n\n"
    f"(All values are global statistics computed over the full available series; timeseries were not filtered by any incident window.)"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  count           p95          mean           max
0          carts         cpu     25  1.012805e+01  4.082323e+00  1.129794e+01
1          carts  latency-50     25  1.535214e-02  1.226511e-02  1.859753e-02
2          carts  latency-90     25  2.557368e-02  2.347247e-02  3.914715e-02
3          carts         mem     25  2.104518e+08  2.055066e+08  2.106542e+08
4          carts      socket     25  1.108667e+01  1.015067e+01  1.145000e+01
..           ...         ...    ...           ...           ...           ...
45  queue-master         mem     25  2.637528e+08  2.629248e+08  2.638537e+08
46  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00
47      rabbitmq         cpu     25  8.785281e-01  8.381490e-01  1.237500e+00
48      rabbitmq      diskio     25  8.004970e+03  3.064678e+03  6.661073e+04
49      rabbitmq         mem     25  9.229748e+07  9.022260e+07  9.257151e+07

[50 rows x 6 columns],               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing df if present, otherwise load
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
else:
    # ensure correct types
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (thresholds computed BEFORE window filtering)
global_p95 = (
    df.groupby(['cmdb_id', 'kpi_name'])['value']
      .quantile(0.95)
      .reset_index(name='global_p95')
)

# 2) Merge global_p95 into full dataframe (so each sample has its series' global threshold)
df2 = df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Define incident window (UTC) and filter points in that window
start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
end   = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')
df_window = df2[(df2['timestamp'] >= start) & (df2['timestamp'] <= end)].copy()

# 4) Mark anomaly points where value >= global_p95
df_window['is_anom'] = df_window['value'] >= df_window['global_p95']

# 5) Aggregate per (cmdb_id, kpi_name) within the window
agg_window = (
    df_window
      .groupby(['cmdb_id', 'kpi_name'])
      .agg(
          total_points_in_window=('value', 'count'),
          anomaly_count=('is_anom', 'sum'),
          max_value_in_window=('value', 'max')
      )
      .reset_index()
)

# 6) Earliest anomaly timestamp per series (if any)
earliest_anom = (
    df_window[df_window['is_anom']]
      .groupby(['cmdb_id', 'kpi_name'])['timestamp']
      .min()
      .reset_index(name='earliest_anomaly_timestamp')
)

# 7) Combine aggregates with global_p95 and earliest anomaly ts
result = (
    agg_window
      .merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')
      .merge(earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')
)

# Format earliest_anomaly_timestamp as ISO UTC string, keep NaN if none
if 'earliest_anomaly_timestamp' in result.columns:
    result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Sort by anomaly_count descending and limit to top 20
result_table = result.sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False]).head(20)

# 8) Small summary: components with at least one anomaly and number of distinct anomalous KPIs
anomaly_summary_by_component = (
    df_window[df_window['is_anom']]
      .groupby('cmdb_id')['kpi_name']
      .nunique()
      .reset_index(name='distinct_anomalous_kpi_count')
      .sort_values('distinct_anomalous_kpi_count', ascending=False)
      .head(20)
)

# Keep outputs compact: show the result_table (selected columns) and the anomaly_summary_by_component
result_table[['cmdb_id','kpi_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']], anomaly_summary_by_component

```
Out[2]:
```
```python
# Build a plain-English summary string from prior results in the IPython state
try:
    result_table
    anomaly_summary_by_component
except NameError:
    # Fallback: load and recompute minimal data if previous variables aren't present
    import pandas as pd
    df = pd.read_csv('metric.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    global_p95 = df.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index(name='global_p95')
    start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
    end   = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')
    df2 = df.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')
    win = df2[(df2['timestamp']>=start)&(df2['timestamp']<=end)].copy()
    win['is_anom'] = win['value'] >= win['global_p95']
    agg = (win.groupby(['cmdb_id','kpi_name'])
             .agg(total_points_in_window=('value','count'),
                  anomaly_count=('is_anom','sum'),
                  max_value_in_window=('value','max')).reset_index())
    earliest = win[win['is_anom']].groupby(['cmdb_id','kpi_name'])['timestamp'].min().reset_index(name='earliest_anomaly_timestamp')
    result_table = agg.merge(global_p95, on=['cmdb_id','kpi_name'], how='left').merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    if 'earliest_anomaly_timestamp' in result_table.columns:
        result_table['earliest_anomaly_timestamp'] = result_table['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    result_table = result_table.sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20)
    anomaly_summary_by_component = (win[win['is_anom']].groupby('cmdb_id')['kpi_name'].nunique().reset_index(name='distinct_anomalous_kpi_count').sort_values('distinct_anomalous_kpi_count',ascending=False).head(20))

# Prepare top anomaly rows summary (compact)
top_rows = result_table[result_table['anomaly_count']>0].sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(10)
lines = []
for _, r in top_rows.iterrows():
    lines.append(
        f"- {r['cmdb_id']} | {r['kpi_name']}: anomalies={int(r['anomaly_count'])}, "
        f"points_in_window={int(r['total_points_in_window'])}, earliest={r.get('earliest_anomaly_timestamp', 'n/a')}, "
        f"max_in_window={float(r['max_value_in_window']):.6g}, global_p95={float(r['global_p95']):.6g}"
    )

# Components with at least one anomalous KPI (compact list)
comp_lines = []
for _, r in anomaly_summary_by_component.head(20).iterrows():
    comp_lines.append(f"- {r['cmdb_id']}: {int(r['distinct_anomalous_kpi_count'])} distinct anomalous KPIs")

summary = (
    "Anomaly detection summary (window 2024-01-20 16:25:45 to 2024-01-20 16:55:45 UTC):\n\n"
    f"Total components with ≥1 anomaly: {anomaly_summary_by_component.shape[0]}\n\n"
    "Top anomalous series (ordered by anomaly_count):\n"
    + ("\n".join(lines) if lines else " - No anomalies detected in the window") + "\n\n"
    "Components with anomalous KPIs and count of distinct anomalous KPIs:\n"
    + ("\n".join(comp_lines) if comp_lines else " - None")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
14          catalogue      socket  5.000000e+00                      25             25       2024-01-20T16:28:00Z         5.000000e+00
17       catalogue-db      diskio  0.000000e+00                      25             25       2024-01-20T16:28:00Z         0.000000e+00
19       catalogue-db      socket  4.000000e+00                      25             25       2024-01-20T16:28:00Z         4.000000e+00
35          orders-db      socket  7.000000e+00                      25             25       2024-01-20T16:28:00Z         7.000000e+00
46       queue-master      socket  3.000000e+00                      25             25       2024-01-20T16:28:00Z         3.000000e+00
50           rabbitmq      socket  1.100000e+01                      25             25       2024-01-20T16:28:00Z         1.100000e+01
52  rabbitmq-exporter         mem  4.485120e+06                      25             25       2024-01-20T16:28:00Z         4.485120e+06
53  rabbitmq-exporter      socket  1.000000e+00                      25             25       2024-01-20T16:28:00Z         1.000000e+00
57         session-db      socket  3.000000e+00                      25             25       2024-01-20T16:28:00Z         3.000000e+00
37            payment      diskio  0.000000e+00                      24             24       2024-01-20T16:29:00Z         0.000000e+00
9            carts-db      socket  6.000000e+00                      25             22       2024-01-20T16:31:00Z         6.000000e+00
41            payment      socket  5.000000e+00                      25             10       2024-01-20T16:43:00Z         5.000000e+00
68               user      socket  1.500000e+01                      25              6       2024-01-20T16:47:00Z         1.500000e+01
73            user-db      socket  8.000000e+00                      25              6       2024-01-20T16:47:00Z         8.000000e+00
0               carts         cpu  1.012805e+01                      25              2       2024-01-20T16:28:00Z         1.129794e+01
1               carts  latency-50  1.535214e-02                      25              2       2024-01-20T16:28:00Z         1.859753e-02
2               carts  latency-90  2.557368e-02                      25              2       2024-01-20T16:28:00Z         3.914715e-02
3               carts         mem  2.104518e+08                      25              2       2024-01-20T16:51:00Z         2.106542e+08
4               carts      socket  1.108667e+01                      25              2       2024-01-20T16:46:00Z         1.145000e+01
5               carts    workload  8.545327e+00                      25              2       2024-01-20T16:30:00Z         8.569883e+00,               cmdb_id  distinct_anomalous_kpi_count
7             payment                             7
2           catalogue                             6
0               carts                             6
5              orders                             6
13               user                             6
12           shipping                             6
4           front-end                             6
6           orders-db                             4
3        catalogue-db                             4
1            carts-db                             4
8        queue-master                             4
11         session-db                             4
9            rabbitmq                             4
14            user-db                             4
10  rabbitmq-exporter                             3)```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse kernel where possible; load log.csv and error_logs.csv
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Parse timestamps as UTC datetimes per rules
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Ensure numeric value in log.csv
df_log['value'] = pd.to_numeric(df_log.get('value', pd.Series()), errors='coerce')

# Incident window (UTC)
start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
end   = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')

# Filter to window and store full filtered tables in kernel variables for follow-up
logs_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()
error_logs_window = df_err[(df_err['timestamp'] >= start) & (df_err['timestamp'] <= end)].copy()

# (A) Aggregates from log.csv
if logs_window.empty:
    logs_agg_top50 = pd.DataFrame({'indicator': ['NO_ROWS_IN_WINDOW_FOR_log.csv']})
    logs_empty_indicator = "NO_ROWS_IN_WINDOW_FOR_log.csv"
else:
    # Group counts and max value
    logs_grp = (
        logs_window
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            count=('value', 'count'),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )

    # Earliest timestamp where value > 0
    pos = logs_window[logs_window['value'] > 0]
    if not pos.empty:
        earliest_pos = (
            pos.groupby(['cmdb_id', 'log_name'], dropna=False)['timestamp']
               .min()
               .reset_index(name='earliest_timestamp')
        )
        earliest_pos['earliest_timestamp'] = earliest_pos['earliest_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        earliest_pos = pd.DataFrame(columns=['cmdb_id', 'log_name', 'earliest_timestamp'])

    # Merge earliest positive timestamps (may be missing for some groups)
    logs_agg = logs_grp.merge(earliest_pos, on=['cmdb_id', 'log_name'], how='left')

    # Sort by count desc and limit to top 50
    logs_agg_top50 = logs_agg.sort_values('count', ascending=False).head(50).reset_index(drop=True)
    logs_empty_indicator = None

# (B) Aggregates from error_logs.csv
if error_logs_window.empty:
    error_agg_top50 = pd.DataFrame({'indicator': ['NO_ROWS_IN_WINDOW_FOR_error_logs.csv']})
    errors_empty_indicator = "NO_ROWS_IN_WINDOW_FOR_error_logs.csv"
else:
    # Group by full message, count and earliest timestamp
    err_grp = (
        error_logs_window
        .groupby(['cmdb_id', 'message'], dropna=False)
        .agg(
            count=('timestamp', 'count'),
            earliest_timestamp=('timestamp', 'min')
        )
        .reset_index()
    )

    # Truncate message to first 200 chars for display
    err_grp['message_truncated'] = err_grp['message'].astype(str).str.slice(0, 200)

    # Format earliest timestamp as ISO UTC
    err_grp['earliest_timestamp'] = err_grp['earliest_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    # Keep requested columns and sort by count desc, limit 50
    error_agg_top50 = (
        err_grp[['cmdb_id', 'message_truncated', 'count', 'earliest_timestamp']]
        .rename(columns={'message_truncated': 'message'})
        .sort_values('count', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )
    errors_empty_indicator = None

# Return compact aggregates (and full filtered tables are stored in logs_window and error_logs_window)
logs_agg_top50, error_agg_top50

```
Out[3]:
```
```python
# Build a concise plain-English summary using existing kernel variables
try:
    logs_agg_top50
    error_agg_top50
    logs_window
    error_logs_window
except NameError:
    # If variables are missing, load minimal data to create a fallback summary
    import pandas as pd
    logs_window = pd.read_csv('log.csv')
    logs_window['timestamp'] = pd.to_datetime(logs_window['timestamp'], unit='s', utc=True)
    logs_window = logs_window[(logs_window['timestamp'] >= '2024-01-20T16:25:45Z') & (logs_window['timestamp'] <= '2024-01-20T16:55:45Z')]
    logs_window['value'] = pd.to_numeric(logs_window.get('value', pd.Series()), errors='coerce')
    logs_agg_top50 = (logs_window.groupby(['cmdb_id','log_name'], dropna=False)
                      .agg(count=('value','count'), max_value_in_window=('value','max')).reset_index()).sort_values('count', ascending=False).head(50)
    error_logs_window = pd.read_csv('error_logs.csv')
    error_logs_window['timestamp'] = pd.to_datetime(error_logs_window['timestamp'], unit='s', utc=True)
    error_logs_window = error_logs_window[(error_logs_window['timestamp'] >= '2024-01-20T16:25:45Z') & (error_logs_window['timestamp'] <= '2024-01-20T16:55:45Z')]
    if error_logs_window.empty:
        error_agg_top50 = None
    else:
        err = (error_logs_window.groupby(['cmdb_id','message'], dropna=False)
               .agg(count=('timestamp','count'), earliest_timestamp=('timestamp','min')).reset_index())
        err['message'] = err['message'].astype(str).str.slice(0,200)
        err['earliest_timestamp'] = err['earliest_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        error_agg_top50 = err.sort_values('count', ascending=False).head(50)

# Compose summary
num_log_groups = 0 if (logs_agg_top50 is None) else int(logs_agg_top50.shape[0])
num_error_rows = 0 if (error_logs_window is None) else int(error_logs_window.shape[0])

# Prepare top examples from logs_agg_top50
examples = []
if num_log_groups > 0:
    sample = logs_agg_top50.head(6)[['cmdb_id','log_name','count','max_value_in_window','earliest_timestamp']].fillna('NaN')
    for _, r in sample.iterrows():
        examples.append(f"{r['cmdb_id']} | {r['log_name']}: count={int(r['count'])}, max={r['max_value_in_window']}, earliest_positive={r['earliest_timestamp']}")
else:
    examples.append("No log rows in the window.")

# Error logs summary
if num_error_rows == 0:
    error_msg = "error_logs.csv: NO ROWS IN WINDOW (no error log entries between 2024-01-20T16:25:45Z and 2024-01-20T16:55:45Z)."
else:
    # show top 3 error groups
    top_err = error_agg_top50.head(3)
    err_lines = []
    for _, r in top_err.iterrows():
        err_lines.append(f"{r['cmdb_id']} | {r['message'][:200]} : count={int(r['count'])}, earliest={r['earliest_timestamp']}")
    error_msg = "error_logs.csv top errors:\n" + "\n".join(err_lines)

summary = (
    "Log & error-log window summary (2024-01-20 16:25:45 to 2024-01-20 16:55:45 UTC):\n\n"
    f"1) log.csv (filtered): {num_log_groups} distinct (cmdb_id, log_name) groups found in the window.\n"
    "- Observation: Many groups have the full window sample count (25) indicating per-minute coverage across the window.\n"
    "- Important pattern: 'log.error_count' groups often have max_value_in_window = 0 (no recorded errors),\n"
    "  while 'log.row_count' groups show positive maxima (example rows below).\n\n"
    "Top example log groups (cmdb_id | log_name: count, max, earliest_positive):\n"
    + "\n".join(f"  {s}" for s in examples) + "\n\n"
    f"2) error_logs.csv (filtered): {num_error_rows} rows found.\n"
    f"   {error_msg}\n\n"
    "Stored variables for follow-up in kernel:\n"
    "- Full filtered log rows: logs_window\n"
    "- Full filtered error log rows: error_logs_window\n\n"
    "Notes: Outputs are compact summaries. Use logs_window.head(n) or error_logs_window.head(n) for detailed inspection."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id         log_name  count  max_value_in_window    earliest_timestamp
0    catalogue  log.error_count     25                    0                   NaN
1    catalogue    log.row_count     25                  170  2024-01-20T16:28:00Z
2       orders    log.row_count     25                  134  2024-01-20T16:28:00Z
3       orders  log.error_count     25                    0                   NaN
4    front-end    log.row_count     25                 2084  2024-01-20T16:28:00Z
..         ...              ...    ...                  ...                   ...
17       carts    log.row_count     22                  150  2024-01-20T16:29:00Z
18  session-db  log.error_count      5                    0                   NaN
19  session-db    log.row_count      5                    5  2024-01-20T16:29:00Z
20     user-db  log.error_count      1                    0                   NaN
21     user-db    log.row_count      1                    1  2024-01-20T16:45:00Z

[22 rows x 5 columns],                               indicator
0  NO_ROWS_IN_WINDOW_FOR_error_logs.csv)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing variables if present; otherwise load trace.csv
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Parse timestamps as UTC datetimes and ensure numeric values
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (thresholds before filtering)
global_p95 = (
    trace_df
      .groupby(['cmdb_id', 'trace_name'])['value']
      .quantile(0.95)
      .reset_index(name='global_p95')
)

# 2) Merge global_p95 into the trace dataframe
trace_with_threshold = trace_df.merge(global_p95, on=['cmdb_id', 'trace_name'], how='left')

# 3) Define incident window (UTC) and filter trace points to the window
start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
end   = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')
trace_window = trace_with_threshold[(trace_with_threshold['timestamp'] >= start) & (trace_with_threshold['timestamp'] <= end)].copy()

# Mark anomaly points where value >= global_p95
trace_window['is_anom'] = trace_window['value'] >= trace_window['global_p95']

# Aggregate per (cmdb_id, trace_name) within the window
if trace_window.empty:
    # empty indicators
    result_table = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'])
    anomaly_summary_by_component = pd.DataFrame(columns=['cmdb_id','distinct_anomalous_trace_count'])
else:
    agg_window = (
        trace_window
          .groupby(['cmdb_id', 'trace_name'], dropna=False)
          .agg(
              total_points_in_window=('value', 'count'),
              anomaly_count=('is_anom', 'sum'),
              max_value_in_window=('value', 'max')
          )
          .reset_index()
    )

    # Earliest anomaly timestamp per series (if any)
    earliest_anom = (
        trace_window[trace_window['is_anom']]
          .groupby(['cmdb_id', 'trace_name'])['timestamp']
          .min()
          .reset_index(name='earliest_anomaly_timestamp')
    )

    # Combine aggregates with global_p95 and earliest anomaly timestamp
    result_table = (
        agg_window
          .merge(global_p95, on=['cmdb_id', 'trace_name'], how='left')
          .merge(earliest_anom, on=['cmdb_id', 'trace_name'], how='left')
    )

    # Format earliest_anomaly_timestamp as ISO UTC string (Z) where present
    if 'earliest_anomaly_timestamp' in result_table.columns:
        result_table['earliest_anomaly_timestamp'] = result_table['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    # Sort by anomaly_count desc and limit to top 20
    result_table = result_table.sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False]).head(20)

    # Summary: components with at least one trace anomaly and count of distinct anomalous trace_names
    anomaly_summary_by_component = (
        trace_window[trace_window['is_anom']]
          .groupby('cmdb_id')['trace_name']
          .nunique()
          .reset_index(name='distinct_anomalous_trace_count')
          .sort_values('distinct_anomalous_trace_count', ascending=False)
          .head(20)
    )

# Keep full filtered trace rows in kernel variable 'trace_window' for follow-up (but do NOT display it here)
# Display compact aggregated results only:
result_table[['cmdb_id','trace_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']], anomaly_summary_by_component

```
Out[4]:
```
```python
# Summarize trace analysis result in plain English and display as a single string.
try:
    trace_window
    result_table
    anomaly_summary_by_component
except NameError:
    # If variables missing, set fallback empty indicators
    trace_window = None
    result_table = None
    anomaly_summary_by_component = None

start_iso = "2024-01-20T16:25:45Z"
end_iso = "2024-01-20T16:55:45Z"

if trace_window is None:
    summary = (
        f"Trace anomaly detection (window {start_iso} to {end_iso} UTC):\n\n"
        "No trace data available in the kernel to analyze."
    )
else:
    if trace_window.empty:
        summary = (
            f"Trace anomaly detection (window {start_iso} to {end_iso} UTC):\n\n"
            "- No trace points were found in the incident window; therefore no anomalies were detected.\n"
            "- The full (empty) filtered trace table is stored in the kernel variable 'trace_window' for follow-up."
        )
    else:
        # Some trace rows exist; check result_table for anomalies
        if result_table is None or result_table.empty:
            total_points = int(trace_window.shape[0])
            unique_series = int(trace_window.groupby(['cmdb_id','trace_name']).ngroups)
            summary = (
                f"Trace anomaly detection (window {start_iso} to {end_iso} UTC):\n\n"
                f"- {total_points} trace points across {unique_series} unique (cmdb_id, trace_name) series were examined.\n"
                "- No anomaly points met or exceeded the global P95 thresholds for their series in this window.\n"
                "- Full filtered trace rows are stored in the kernel variable 'trace_window' for follow-up."
            )
        else:
            # Some anomalies detected (not expected here but handle generically)
            top_series = result_table.head(5)[['cmdb_id','trace_name','anomaly_count']].to_dict('records')
            comp_lines = "; ".join([f"{r['cmdb_id']}:{r['trace_name']}({int(r['anomaly_count'])})" for r in top_series])
            summary = (
                f"Trace anomaly detection (window {start_iso} to {end_iso} UTC):\n\n"
                "- Anomalies were detected in some trace series. Top examples (cmdb_id:trace_name(count)):\n"
                f"  {comp_lines}\n"
                "- Full aggregated results are in 'result_table' and full filtered traces are in 'trace_window'."
            )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, trace_name, global_p95, total_points_in_window, anomaly_count, earliest_anomaly_timestamp, max_value_in_window]
Index: [], Empty DataFrame
Columns: [cmdb_id, distinct_anomalous_trace_count]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing metric dataframe 'df' if present; otherwise load metric.csv
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# Ensure timestamp and value columns are correctly typed
if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Filter the full series for component 'payment' (for global thresholds we still use the full series of this component)
payment_full = df[df['cmdb_id'] == 'payment'].copy()

# Compute global P95 per kpi_name using the full payment series (thresholds computed BEFORE window filtering)
global_p95 = payment_full.groupby('kpi_name')['value'].quantile(0.95).reset_index(name='global_p95')

# Merge global_p95 into payment_full so each sample has its series threshold (keeps full series context)
payment_full = payment_full.merge(global_p95, on='kpi_name', how='left')

# Define incident window (UTC) and filter payment points to the window; store full filtered rows in kernel variable
start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
end   = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')
payment_window = payment_full[(payment_full['timestamp'] >= start) & (payment_full['timestamp'] <= end)].copy()

# Detect anomaly points where value >= global_p95
payment_window['is_anom'] = payment_window['value'] >= payment_window['global_p95']

# Aggregate per kpi_name within the window
if payment_window.empty:
    result_table = pd.DataFrame(columns=['kpi_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window'])
else:
    agg = (
        payment_window
          .groupby('kpi_name', dropna=False)
          .agg(
              total_points_in_window=('value', 'count'),
              anomaly_count=('is_anom', 'sum'),
              max_value_in_window=('value', 'max')
          )
          .reset_index()
    )

    # Earliest anomaly timestamp per kpi_name (if any)
    earliest = (
        payment_window[payment_window['is_anom']]
          .groupby('kpi_name')['timestamp']
          .min()
          .reset_index(name='earliest_anomaly_timestamp')
    )

    # Combine with global_p95
    result_table = (
        agg
          .merge(global_p95, on='kpi_name', how='left')
          .merge(earliest, on='kpi_name', how='left')
    )

    # Format earliest_anomaly_timestamp as ISO UTC string
    if 'earliest_anomaly_timestamp' in result_table.columns:
        result_table['earliest_anomaly_timestamp'] = result_table['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    # Reorder columns as requested
    result_table = result_table[['kpi_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']]

    # Sort by anomaly_count desc and limit to top 20
    result_table = result_table.sort_values(['anomaly_count','total_points_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Return compact aggregated table and the shape of the stored full filtered payment rows (do not print full raw series)
result_table, payment_window.shape

```
Out[5]:
```
```python
# Build a plain-English summary string from existing kernel results for the 'payment' component.
try:
    result_table
    payment_window
except NameError:
    import pandas as pd
    df = pd.read_csv('metric.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    payment_full = df[df['cmdb_id']=='payment'].copy()
    global_p95 = payment_full.groupby('kpi_name')['value'].quantile(0.95).reset_index(name='global_p95')
    payment_full = payment_full.merge(global_p95, on='kpi_name', how='left')
    start = pd.Timestamp('2024-01-20 16:25:45', tz='UTC')
    end = pd.Timestamp('2024-01-20 16:55:45', tz='UTC')
    payment_window = payment_full[(payment_full['timestamp']>=start)&(payment_full['timestamp']<=end)].copy()
    payment_window['is_anom'] = payment_window['value'] >= payment_window['global_p95']
    agg = (payment_window.groupby('kpi_name', dropna=False)
           .agg(total_points_in_window=('value','count'),
                anomaly_count=('is_anom','sum'),
                max_value_in_window=('value','max')).reset_index())
    earliest = (payment_window[payment_window['is_anom']].groupby('kpi_name')['timestamp'].min().reset_index(name='earliest_anomaly_timestamp'))
    result_table = agg.merge(global_p95, on='kpi_name', how='left').merge(earliest, on='kpi_name', how='left')
    if 'earliest_anomaly_timestamp' in result_table.columns:
        result_table['earliest_anomaly_timestamp'] = result_table['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    result_table = result_table[['kpi_name','global_p95','total_points_in_window','anomaly_count','earliest_anomaly_timestamp','max_value_in_window']].sort_values(['anomaly_count','total_points_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Compose concise summary
lines = []
lines.append("Payment component anomaly summary")
lines.append("Window: 2024-01-20 16:25:45 to 2024-01-20 16:55:45 UTC")
lines.append(f"Filtered payment rows stored in kernel variable 'payment_window' (shape: {payment_window.shape})")
lines.append("")
lines.append("Per-KPI summary (kpi_name: anomalies / points_in_window, earliest_anomaly, max_in_window, global_p95):")

for _, r in result_table.iterrows():
    k = r['kpi_name']
    anomalies = int(r['anomaly_count']) if not pd.isna(r['anomaly_count']) else 0
    total = int(r['total_points_in_window']) if not pd.isna(r['total_points_in_window']) else 0
    earliest = r['earliest_anomaly_timestamp'] if (pd.notna(r.get('earliest_anomaly_timestamp'))) else "None"
    maxv = float(r['max_value_in_window']) if not pd.isna(r['max_value_in_window']) else None
    gp95 = float(r['global_p95']) if not pd.isna(r['global_p95']) else None
    lines.append(f"- {k}: {anomalies}/{total} anomalies, earliest={earliest}, max={maxv:.6g}, global_p95={gp95:.6g}")

lines.append("")
lines.append("Highlights:")
# Determine top contributors
if 'diskio' in result_table['kpi_name'].values:
    diskio_row = result_table[result_table['kpi_name']=='diskio'].iloc[0]
    if int(diskio_row['anomaly_count'])>0:
        lines.append("- diskio: all window points were at or above global P95 (anomaly_count equals points in window). Values are 0, indicating p95=0.")
# socket
if 'socket' in result_table['kpi_name'].values:
    sock_row = result_table[result_table['kpi_name']=='socket'].iloc[0]
    lines.append(f"- socket: {int(sock_row['anomaly_count'])} anomalous points (max {float(sock_row['max_value_in_window']):.6g}, p95={float(sock_row['global_p95']):.6g}).")
lines.append("- Other KPIs (cpu, latency-50, latency-90, mem, workload) show only 2 anomaly points each in the window.")
lines.append("")
lines.append("Interpretation: diskio and socket exhibit the most widespread anomalies in the payment component during the window; other KPIs show minor/isolated exceedances of their global P95. Use 'payment_window' for detailed inspection of raw points if needed.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name    global_p95  total_points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window
0      diskio  0.000000e+00                      24             24       2024-01-20T16:29:00Z         0.000000e+00
1      socket  5.000000e+00                      25             10       2024-01-20T16:43:00Z         5.000000e+00
2         cpu  1.454735e-01                      25              2       2024-01-20T16:28:00Z         2.129017e-01
3  latency-50  6.813004e-02                      25              2       2024-01-20T16:46:00Z         7.266746e-02
4  latency-90  2.204214e-01                      25              2       2024-01-20T16:44:00Z         2.211053e-01
5         mem  5.328541e+06                      25              2       2024-01-20T16:35:00Z         5.395115e+06
6    workload  2.131010e+00                      25              2       2024-01-20T16:36:00Z         2.151067e+00, (174, 6))```
```